# Experimento 2

# 1. Configuración Inicial

In [1]:
!pip install -q open_clip_torch torch torchvision opencv-python-headless pillow

import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import open_clip
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
Mounted at /content/drive
Usando dispositivo: cuda


# 2. Cargar el modelo BioCLIP y preparar el Dataset con Smart Crop

In [4]:
import os

# Ruta exacta a la carpeta dataset
dataset_path = "/content/drive/MyDrive/dataset"

# 1. Ver qué hay inmediatamente dentro de la carpeta
elementos = os.listdir(dataset_path)

print(f"Archivos y carpetas dentro de '{dataset_path}':")
for elemento in elementos:
    ruta_completa = os.path.join(dataset_path, elemento)
    if os.path.isdir(ruta_completa):
        print(f"📁 [Carpeta] {elemento}")
    else:
        print(f"📄 [Archivo] {elemento}")

Archivos y carpetas dentro de '/content/drive/MyDrive/dataset':
📁 [Carpeta] Seed dataset extraction


In [6]:
DATASET_DIR = "/content/drive/MyDrive/dataset/Seed dataset extraction"

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'hf-hub:imageomics/bioclip'
)
model = model.to(device)
model.eval()

# Definir la función de Smart Crop con OpenCV integrada al pipeline
def smart_crop_pil(pil_img, padding_ratio=0.2):
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    width, height = pil_img.size
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        cx, cy = x + w // 2, y + h // 2
        max_side = max(w, h)
        crop_size = int(max_side * (1 + padding_ratio))

        left = max(0, cx - crop_size // 2)
        top = max(0, cy - crop_size // 2)
        right = min(width, cx + crop_size // 2)
        bottom = min(height, cy + crop_size // 2)
        return pil_img.crop((left, top, right, bottom))
    else:
        min_dim = min(width, height)
        left = (width - min_dim) / 2
        top = (height - min_dim) / 2
        right = (width + min_dim) / 2
        bottom = (height + min_dim) / 2
        return pil_img.crop((left, top, right, bottom))

# Dataset personalizado en PyTorch
class SeedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []
        for cls_name in self.classes:
            cls_folder = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_folder, fname), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
            cropped_img = smart_crop_pil(img)
            if self.transform:
                tensor_img = self.transform(cropped_img)
            return tensor_img, label

dataset = SeedDataset(root_dir=DATASET_DIR, transform=preprocess_val)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
print(f"Total de imágenes: {len(dataset)} | Clases totales: {len(dataset.classes)}")

Total de imágenes: 4497 | Clases totales: 88


# 3. Extraer Embeddings y entrenar el Clasificador Lineal (Linear Probing)

In [7]:
# 1. Extraer Embeddings de todas las imágenes
embeddings_list = []
labels_list = []

print("Extrayendo embeddings con BioCLIP...")
with torch.no_grad():
    for imgs, labels in tqdm(dataloader):
        imgs = imgs.to(device)
        features = model.encode_image(imgs)
        features /= features.norm(dim=-1, keepdim=True) # Normalizar vectores

        embeddings_list.append(features.cpu())
        labels_list.append(labels)

X = torch.cat(embeddings_list, dim=0)
y = torch.cat(labels_list, dim=0)

print(f"\nMatriz de Embeddings creada: {X.shape}") # Debería ser [4000, 512] aprox.

# 2. Definir y entrenar la Capa Lineal (Classifier)
input_dim = X.shape[1]
num_classes = len(dataset.classes)

classifier = nn.Linear(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-2)

# Crear dataset de tensores extraídos
from torch.utils.data import TensorDataset
emb_dataset = TensorDataset(X, y)
train_loader = DataLoader(emb_dataset, batch_size=64, shuffle=True)

# Loop de entrenamiento (15 épocas toman ~10 segundos)
epochs = 15
print("\nIniciando entrenamiento de la capa lineal...")
classifier.train()

for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    total = 0

    for emb_batch, y_batch in train_loader:
        emb_batch, y_batch = emb_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = classifier(emb_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()

    acc = 100. * correct / total
    print(f"Época [{epoch+1}/{epochs}] - Loss: {total_loss/len(train_loader):.4f} - Precisión Train: {acc:.2f}%")

Extrayendo embeddings con BioCLIP...


100%|██████████| 141/141 [23:24<00:00,  9.96s/it]



Matriz de Embeddings creada: torch.Size([4497, 512])

Iniciando entrenamiento de la capa lineal...
Época [1/15] - Loss: 4.3612 - Precisión Train: 14.63%
Época [2/15] - Loss: 4.1212 - Precisión Train: 30.51%
Época [3/15] - Loss: 3.9005 - Precisión Train: 41.94%
Época [4/15] - Loss: 3.6933 - Precisión Train: 48.48%
Época [5/15] - Loss: 3.4998 - Precisión Train: 56.04%
Época [6/15] - Loss: 3.3134 - Precisión Train: 61.77%
Época [7/15] - Loss: 3.1375 - Precisión Train: 66.64%
Época [8/15] - Loss: 2.9680 - Precisión Train: 69.78%
Época [9/15] - Loss: 2.8123 - Precisión Train: 71.65%
Época [10/15] - Loss: 2.6632 - Precisión Train: 73.60%
Época [11/15] - Loss: 2.5260 - Precisión Train: 75.36%
Época [12/15] - Loss: 2.3948 - Precisión Train: 76.43%
Época [13/15] - Loss: 2.2725 - Precisión Train: 77.67%
Época [14/15] - Loss: 2.1628 - Precisión Train: 79.07%
Época [15/15] - Loss: 2.0562 - Precisión Train: 80.23%


# 4. Guardar los pesos entrenados para usarlos en la App

In [8]:
# Guardar el estado de la capa lineal
checkpoint = {
    'state_dict': classifier.state_dict(),
    'classes': dataset.classes,
    'class_to_idx': dataset.class_to_idx
}

torch.save(checkpoint, "seed_classifier.pth")
print("¡Modelo guardado exitosamente como 'seed_classifier.pth'!")

¡Modelo guardado exitosamente como 'seed_classifier.pth'!


In [9]:
import shutil

# Definir la ruta de destino dentro de tu Google Drive
drive_destination = "/content/drive/MyDrive/seed_classifier.pth"

# Copiar el archivo local al Drive
shutil.copy("seed_classifier.pth", drive_destination)
print(f"¡El modelo ha sido copiado y guardado en tu Google Drive en: {drive_destination}!")

¡El modelo ha sido copiado y guardado en tu Google Drive en: /content/drive/MyDrive/seed_classifier.pth!
